# RAG Knowledge Base Initialization

This notebook builds your RAG system knowledge base:
1. **Load** PDFs from `data/raw_papers/`
2. **Chunk** them into manageable pieces
3. **Store** chunks in `data/chunks.jsonl` (ready for embeddings)
4. **Query** and verify the stored knowledge

Run cells sequentially from top to bottom.

## 0: Setup & Load Configuration

In [5]:
import logging
from pathlib import Path
from datetime import datetime

from backend.core.config import settings
from backend.services.paper_processor import PaperProcessor
from backend.services.document_loader import DocumentLoader
from backend.services.chunk_store import ChunkStore
from backend.services.paper_registry import PaperRegistry

logging.basicConfig(
    level=settings.log_level,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

print("✓ Imports successful (kernel restarted)")
print(f"\n--- Configuration ---")
print(f"Data directory:    {settings.data_dir}")
print(f"Raw papers:        {settings.raw_papers_dir}")
print(f"Chunks file:       {settings.chunks_file}")
print(f"Manifest:          {settings.manifest_path}")
print(f"\n--- Chunking Config ---")
print(f"Chunk size:        {settings.chunk_size}")
print(f"Chunk overlap:     {settings.chunk_overlap}")
print(f"Log level:         {settings.log_level}")

✓ Imports successful (kernel restarted)

--- Configuration ---
Data directory:    d:\Programming\projects\RAG\data
Raw papers:        d:\Programming\projects\RAG\data\raw_papers
Chunks file:       d:\Programming\projects\RAG\data\chunks.jsonl
Manifest:          d:\Programming\projects\RAG\data\papers_manifest.json

--- Chunking Config ---
Chunk size:        1000
Chunk overlap:     200
Log level:         INFO


## 1: Check Prerequisites


In [6]:
pdf_files = list(settings.raw_papers_dir.glob("*.pdf"))

print(f"Checking {settings.raw_papers_dir}...")
print(f"\nFound {len(pdf_files)} PDF files:")

if not pdf_files:
    print("  ❌ NO PDFs FOUND!")
    print(f"\nAdd PDF files to {settings.raw_papers_dir} before continuing.")
else:
    for i, pdf in enumerate(pdf_files, 1):
        size_mb = pdf.stat().st_size / (1024*1024)
        print(f"  {i}. {pdf.name:<40} ({size_mb:.2f} MB)")
    
    print(f"\n✓ Ready to process {len(pdf_files)} PDFs")

Checking d:\Programming\projects\RAG\data\raw_papers...

Found 18 PDF files:
  1. 1906_MC_Layer_Normalization_fo.pdf       (0.65 MB)
  2. accurate_uncertanities_for_dl_using_calibrated_regression.pdf (2.71 MB)
  3. Active_Learning_Framework_for_Improving_Knowledge_Graph_Accuracy.pdf (2.88 MB)
  4. automated_construction_of_theme_specific_kg.pdf (1.29 MB)
  5. dibowski-full-traceability-and-provenance-for-knowledge-graphs.pdf (0.47 MB)
  6. DOC-20251031-WA0004.pdf                  (0.20 MB)
  7. DOC-20251031-WA0005.pdf                  (0.44 MB)
  8. DOC-20251031-WA0006.pdf                  (1.06 MB)
  9. DOC-20251031-WA0007.pdf                  (1.81 MB)
  10. DOC-20251031-WA0008.pdf                  (1.15 MB)
  11. DOC-20251031-WA0009.pdf                  (0.38 MB)
  12. DOC-20251031-WA0010.pdf                  (1.31 MB)
  13. DOC-20251031-WA0011.pdf                  (0.87 MB)
  14. DOC-20251031-WA0012.pdf                  (0.27 MB)
  15. DOC-20251031-WA0013.pdf                  (0.69

## 2: Check Current Status


In [8]:
registry = PaperRegistry(settings.manifest_path)
store = ChunkStore(settings.chunks_file)

print("=" * 60)
print("CURRENT STATUS")
print("=" * 60)

status = registry.get_status()
print(f"\n--- Registry (papers_manifest.json) ---")
print(f"Papers tracked:    {status['total_papers']}")
print(f"Total chunks:      {status['total_chunks']}")

store_stats = store.get_stats()
print(f"\n--- Chunk Store (chunks.jsonl) ---")
print(f"Chunks stored:     {store_stats['total_chunks']}")
print(f"Sources (papers):  {store_stats['total_sources']}")

if status['total_papers'] > 0:
    print(f"\n--- Tracked Papers ---")
    for paper in status['papers']:
        print(f"  • {paper['filename']:<40} ({paper['num_chunks']} chunks)")


2026-03-20 15:49:09,428 - backend.services.paper_registry - INFO - Created new manifest


CURRENT STATUS

--- Registry (papers_manifest.json) ---
Papers tracked:    0
Total chunks:      0

--- Chunk Store (chunks.jsonl) ---
Chunks stored:     0
Sources (papers):  0


## 3: Process Papers (Full Pipeline)

This is the main operation:
- Detects new PDFs (not in manifest)
- Detects updated PDFs (file hash changed)
- For each: Load → Chunk → Store → Register

Idempotent - safe to run multiple times.

In [9]:
processor = PaperProcessor(
    papers_dir=settings.raw_papers_dir,
    manifest_path=settings.manifest_path,
    chunks_file=settings.chunks_file,
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
)

print(f"\n{'='*60}")
print("PROCESSING PAPERS")
print(f"{'='*60}\n")

start_time = datetime.now()
result = processor.process_all()
duration = (datetime.now() - start_time).total_seconds()

print(f"\n{'='*60}")
print("PROCESSING COMPLETE")
print(f"{'='*60}")
print(f"\n--- Results ---")
print(f"New papers processed:     {result['new_processed']}")
print(f"Updated papers processed: {result['updated_processed']}")
print(f"Total chunks created:     {result['total_chunks']}")
print(f"Time elapsed:             {duration:.2f} seconds")

print(f"\n--- Files Updated ---")
if settings.chunks_file.exists():
    size_mb = settings.chunks_file.stat().st_size / (1024*1024)
    print(f"✓ {settings.chunks_file} ({size_mb:.2f} MB, {result['total_chunks']} chunks)")

if settings.manifest_path.exists():
    size_kb = settings.manifest_path.stat().st_size / 1024
    papers_count = result['status']['total_papers']
    print(f"✓ {settings.manifest_path} ({size_kb:.1f} KB, {papers_count} papers)")

2026-03-20 15:49:13,592 - backend.services.paper_registry - INFO - Created new manifest
2026-03-20 15:49:13,595 - backend.services.paper_registry - INFO - Found 18 new papers
2026-03-20 15:49:13,595 - backend.services.paper_processor - INFO - New: 18, Updated: 0
2026-03-20 15:49:13,595 - backend.services.paper_processor - INFO - Processing: 1906_MC_Layer_Normalization_fo.pdf



PROCESSING PAPERS



2026-03-20 15:49:14,079 - backend.services.document_loader - INFO - Loaded 1906_MC_Layer_Normalization_fo.pdf: 22 pages
2026-03-20 15:49:14,082 - backend.services.chunking_service - INFO - Created 82 chunks from 22 documents
2026-03-20 15:49:14,087 - backend.services.chunk_store - INFO - Appended 82 chunks
2026-03-20 15:49:14,091 - backend.services.paper_registry - INFO - Registered: 1906_MC_Layer_Normalization_fo.pdf (82 chunks)
2026-03-20 15:49:14,091 - backend.services.paper_processor - INFO - Processed 1906_MC_Layer_Normalization_fo.pdf: 82 chunks
2026-03-20 15:49:14,093 - backend.services.paper_processor - INFO - Processing: accurate_uncertanities_for_dl_using_calibrated_regression.pdf
2026-03-20 15:49:14,197 - backend.services.document_loader - INFO - Loaded accurate_uncertanities_for_dl_using_calibrated_regression.pdf: 9 pages
2026-03-20 15:49:14,200 - backend.services.chunking_service - INFO - Created 55 chunks from 9 documents
2026-03-20 15:49:14,204 - backend.services.chunk_s


PROCESSING COMPLETE

--- Results ---
New papers processed:     18
Updated papers processed: 0
Total chunks created:     1387
Time elapsed:             3.02 seconds

--- Files Updated ---
✓ d:\Programming\projects\RAG\data\chunks.jsonl (2.15 MB, 1387 chunks)
✓ d:\Programming\projects\RAG\data\papers_manifest.json (3.7 KB, 18 papers)


## 4: Inspect Results


In [11]:
print("\n" + "="*60)
print("KNOWLEDGE BASE SUMMARY")
print("="*60)

# Reload registry to get fresh data from the saved manifest
registry = PaperRegistry(settings.manifest_path)
store_stats = store.get_stats()

status = registry.get_status()

print(f"\n--- Overview ---")
print(f"Papers in knowledge base:  {status['total_papers']}")
print(f"Total chunks available:    {status['total_chunks']}")
print(f"Average chunks per paper:  {status['total_chunks'] // max(1, status['total_papers']):.0f}")

if status['papers']:
    print(f"\n--- Papers Processed ---")
    for i, paper in enumerate(status['papers'], 1):
        chunks = paper['num_chunks']
        processed = paper['processed_at']
        print(f"  {i}. {paper['filename']:<40} {chunks:>4} chunks  ({processed})")

if store_stats['chunks_per_source']:
    print(f"\n--- Chunks by Source ---")
    total_checked = 0
    for source in sorted(store_stats['chunks_per_source'].keys()):
        count = store_stats['chunks_per_source'][source]
        total_checked += count
        print(f"  {Path(source).name:<40} {count:>4} chunks")
    print(f"  {'TOTAL':<40} {total_checked:>4} chunks")

2026-03-20 15:50:44,081 - backend.services.paper_registry - INFO - Loaded manifest: 18 papers



KNOWLEDGE BASE SUMMARY

--- Overview ---
Papers in knowledge base:  18
Total chunks available:    1387
Average chunks per paper:  77

--- Papers Processed ---
  1. 1906_MC_Layer_Normalization_fo.pdf         82 chunks  (2026-03-20T10:19:14.090592)
  2. accurate_uncertanities_for_dl_using_calibrated_regression.pdf   55 chunks  (2026-03-20T10:19:14.212692)
  3. Active_Learning_Framework_for_Improving_Knowledge_Graph_Accuracy.pdf   87 chunks  (2026-03-20T10:19:14.385037)
  4. automated_construction_of_theme_specific_kg.pdf   77 chunks  (2026-03-20T10:19:14.484795)
  5. dibowski-full-traceability-and-provenance-for-knowledge-graphs.pdf   63 chunks  (2026-03-20T10:19:14.585173)
  6. DOC-20251031-WA0004.pdf                    28 chunks  (2026-03-20T10:19:14.642890)
  7. DOC-20251031-WA0005.pdf                    90 chunks  (2026-03-20T10:19:14.754088)
  8. DOC-20251031-WA0006.pdf                    69 chunks  (2026-03-20T10:19:14.875557)
  9. DOC-20251031-WA0007.pdf                    98 chu

## 5: Query Chunks

Load and inspect stored chunks.

In [12]:
print(f"\nLoading chunks from {settings.chunks_file}...\n")

all_chunks = store.load_all()
print(f"✓ Loaded {len(all_chunks)} chunks")

if all_chunks:
    print(f"\n--- Sample Chunks (first 3) ---\n")
    
    for i, chunk in enumerate(all_chunks[:3], 1):
        print(f"Chunk {i}:")
        print(f"  Source:       {chunk.metadata.get('source')}")
        print(f"  Page:         {chunk.metadata.get('page')}")
        print(f"  Chunk index:  {chunk.metadata.get('chunk_index')}")
        print(f"  Size:         {len(chunk.page_content)} chars")
        print(f"  Preview:      {chunk.page_content[:80].replace(chr(10), ' ')}...")
        print()
else:
    print("No chunks found. Process papers in Step 3.")


Loading chunks from d:\Programming\projects\RAG\data\chunks.jsonl...

✓ Loaded 1387 chunks

--- Sample Chunks (first 3) ---

Chunk 1:
  Source:       d:\Programming\projects\RAG\data\raw_papers\1906_MC_Layer_Normalization_fo.pdf
  Page:         0
  Chunk index:  0
  Size:         926 chars
  Preview:      Published in Transactions on Machine Learning Research (02/2024) MC Layer Normal...

Chunk 2:
  Source:       d:\Programming\projects\RAG\data\raw_papers\1906_MC_Layer_Normalization_fo.pdf
  Page:         0
  Chunk index:  1
  Size:         987 chars
  Preview:      applications where shifts in data distribution may occur. Thus, calibrated predi...

Chunk 3:
  Source:       d:\Programming\projects\RAG\data\raw_papers\1906_MC_Layer_Normalization_fo.pdf
  Page:         0
  Chunk index:  2
  Size:         686 chars
  Preview:      and Prediction-Time Batch Normalization. Second, we explore its suitability for ...



## 6: Query by Paper

Get all chunks from a specific paper.

In [13]:
sources = store_stats['chunks_per_source'] if store_stats else {}

if sources:
    # Get first paper
    first_source = sorted(sources.keys())[0]
    paper_name = Path(first_source).name
    chunk_count = sources[first_source]
    
    print(f"\n--- Querying: {paper_name} ---")
    print(f"Expected chunks: {chunk_count}")
    
    chunks = store.get_by_source(first_source)
    
    print(f"Retrieved chunks: {len(chunks)}")
    print(f"Total size: {sum(len(c.page_content) for c in chunks)} characters")
    
    if chunks:
        print(f"\n--- Statistics ---")
        print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")
        print(f"Pages covered: {len(set(c.metadata.get('page') for c in chunks))}")
else:
    print("No sources available. Process papers first.")


--- Querying: 1906_MC_Layer_Normalization_fo.pdf ---
Expected chunks: 82
Retrieved chunks: 82
Total size: 69430 characters

--- Statistics ---
Average chunk size: 847 chars
Pages covered: 22


## 7: Detect Changes

Show what's new or modified.

In [15]:
print(f"\n--- Detecting Changes ---")
print(f"Scanning: {settings.raw_papers_dir}\n")

new_papers = registry.get_new_papers(settings.raw_papers_dir)
updated_papers = registry.get_updated_papers(settings.raw_papers_dir)

print(f"New papers (not processed):      {len(new_papers)}")
for paper in new_papers:
    print(f"  • {paper.name}")

print(f"\nModified papers (hash changed):  {len(updated_papers)}")
for paper in updated_papers:
    print(f"  • {paper.name}")

if not new_papers and not updated_papers:
    print("\n✓ Knowledge base is up to date!")

2026-03-20 13:49:03,735 - backend.services.paper_registry - INFO - Found 18 new papers



--- Detecting Changes ---
Scanning: d:\Programming\projects\RAG\data\raw_papers

New papers (not processed):      18
  • 1906_MC_Layer_Normalization_fo.pdf
  • accurate_uncertanities_for_dl_using_calibrated_regression.pdf
  • Active_Learning_Framework_for_Improving_Knowledge_Graph_Accuracy.pdf
  • automated_construction_of_theme_specific_kg.pdf
  • dibowski-full-traceability-and-provenance-for-knowledge-graphs.pdf
  • DOC-20251031-WA0004.pdf
  • DOC-20251031-WA0005.pdf
  • DOC-20251031-WA0006.pdf
  • DOC-20251031-WA0007.pdf
  • DOC-20251031-WA0008.pdf
  • DOC-20251031-WA0009.pdf
  • DOC-20251031-WA0010.pdf
  • DOC-20251031-WA0011.pdf
  • DOC-20251031-WA0012.pdf
  • DOC-20251031-WA0013.pdf
  • expanding_kg_with_human_in_loop.pdf
  • learning_confidence_for_transformer_based_neural_machine_translation.pdf
  • relational_learning_of_pattern_matching.pdf

Modified papers (hash changed):  0
